In [1]:
import pandas as pd



students = pd.read_csv("data/raw/students.csv")
sessions = pd.read_csv("data/raw/sessions.csv")
quizzes = pd.read_csv("data/raw/quizzes.csv")
completions = pd.read_csv("data/raw/completions.csv")

# convert dates properly
students["enrollment_date"] = pd.to_datetime(students["enrollment_date"])
sessions["session_date"] = pd.to_datetime(sessions["session_date"])
quizzes["quiz_date"] = pd.to_datetime(quizzes["quiz_date"])
completions["last_session_date"] = pd.to_datetime(completions["last_session_date"])

In [2]:
print("students:", students.shape[0])
print("unique students in sessions:", sessions["student_id"].nunique())
print("unique students in quizzes:", quizzes["student_id"].nunique())
print("unique students in completions:", completions["student_id"].nunique())

students: 1500
unique students in sessions: 1427
unique students in quizzes: 1427
unique students in completions: 1500


In [3]:
master = students.merge(completions, on="student_id", how="left")
print("master shape:", master.shape)
print("nulls after merge:", master.isnull().sum())

master shape: (1500, 9)
nulls after merge: student_id           0
enrollment_date      0
course_id            0
age                  0
total_modules        0
modules_completed    0
completed            0
last_session_date    0
dropout_type         0
dtype: int64


In [4]:
sessions_sorted = sessions.sort_values(["student_id", "session_date"])
gaps = sessions_sorted.groupby("student_id")["session_date"].apply(
    lambda x: x.diff().dt.days.max()
).reset_index()
gaps.columns = ["student_id", "max_gap_days"]

master = master.merge(gaps, on="student_id", how="left")
master["max_gap_days"] = master["max_gap_days"].fillna(0)

In [5]:
time_spent = sessions.groupby("student_id")["session_duration_min"].sum().reset_index()
time_spent.columns = ["student_id", "total_time_min"]

master = master.merge(time_spent, on="student_id", how="left")
master["total_time_min"] = master["total_time_min"].fillna(0)

In [6]:
quiz_pct = quizzes.groupby("student_id")["module_number"].nunique().reset_index()
quiz_pct.columns = ["student_id", "quizzes_taken"]

master = master.merge(quiz_pct, on="student_id", how="left")
master["quizzes_taken"] = master["quizzes_taken"].fillna(0)
master["quiz_completion_pct"] = master["quizzes_taken"] / master["total_modules"] * 100

In [7]:
avg_score = quizzes.groupby("student_id")["score"].mean().reset_index()
avg_score.columns = ["student_id", "avg_quiz_score"]

master = master.merge(avg_score, on="student_id", how="left")
master["avg_quiz_score"] = master["avg_quiz_score"].fillna(0)

In [8]:
master.groupby("dropout_type")[["max_gap_days", "total_time_min", "quiz_completion_pct", "avg_quiz_score"]].mean()

,max_gap_days,total_time_min,quiz_completion_pct,avg_quiz_score
dropout_type,,,,
completer,2.574233,934.684663,100.000000,78.655215
explicit_dropout,0.514403,12.362140,10.493827,41.997942
silent_dropout,12.067873,300.307692,49.434389,68.772562


In [9]:
master.groupby("dropout_type")[["max_gap_days", "total_time_min", "quiz_completion_pct", "avg_quiz_score"]].mean()

,max_gap_days,total_time_min,quiz_completion_pct,avg_quiz_score
dropout_type,,,,
completer,2.574233,934.684663,100.000000,78.655215
explicit_dropout,0.514403,12.362140,10.493827,41.997942
silent_dropout,12.067873,300.307692,49.434389,68.772562


In [11]:
master.to_csv("data/processed/master_features.csv", index=False)
